In [1]:
from __future__ import annotations
import argparse
from types import SimpleNamespace
from collections import Counter, defaultdict
import numpy as np
import torch
import gymnasium as gym
from agents.dqn_agent import DQNAgent
from config.training_config import AlgoConfig


In [ ]:
def value_iteration(env, gamma: float = 0.99, theta: float = 1e-8):
    """
    Compute optimal state values and policy for finite MDP with known dynamics env.P.
    env: Taxi-v3 (Gymnasium) with attributes:
         - observation_space.n
         - action_space.n
         - P[s][a] -> list of (prob, next_state, reward, done)
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    print(f"Value Iteration on {env.spec.id}, states: {n_states}, actions: {n_actions}")

    V = np.zeros(n_states, dtype=np.float64)
    while True:
        delta = 0.0
        # Bellman optimality backup
        for s in range(n_states):
            q_sa = []
            for a in range(n_actions):
                q = 0.0
                for p, s2, r, _done in env.P[s][a]:
                    q += p * (r + gamma * V[s2])
                q_sa.append(q)
            v_new = max(q_sa)
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta:
            break

    policy = np.zeros(n_states, dtype=np.int64)
    for s in range(n_states):
        q_sa = []
        for a in range(n_actions):
            q = 0.0
            for p, s2, r, _done in env.P[s][a]:
                q += p * (r + gamma * V[s2])
            q_sa.append(q)
        policy[s] = int(np.argmax(q_sa))
    return policy, V


def make_agent(device: str = "cpu", seed: int = 0) -> DQNAgent:

    def env_factory():
        #return gym.make("Taxi-v3")
        return gym.make("FrozenLake-v1", is_slippery=False) 
    agent = DQNAgent(
        config=AlgoConfig(),
        train_env_factory=env_factory,
        seed=seed,
        device=device,
    )
    return agent


@torch.no_grad()
def trained_greedy_action(agent: DQNAgent, s: int) -> int:
    """
    Query for a single discrete Taxi state 's' and return greedy action.
    We set deterministic=True to avoid epsilon-greedy noise.
    """
    obs = np.array(s, dtype=np.int64)
    act, _extras = agent.predict(obs, deterministic=True)
    return int(act)


def evaluate_correct_action_rate(policy_opt: np.ndarray, agent: DQNAgent):
    """
    Compare greedy actions of the trained agent against the optimal policy over all states.
    Returns summary dict with accuracy and diagnostics.
    """
    n_states = policy_opt.shape[0]
    correct = 0
    mismatches = []
    per_action = Counter()           # how often optimal says action a
    per_action_correct = Counter()   # of those, how often agent matches

    for s in range(n_states):
        a_opt = int(policy_opt[s])
        a_trn = trained_greedy_action(agent, s)

        per_action[a_opt] += 1
        if a_trn == a_opt:
            correct += 1
            per_action_correct[a_opt] += 1
        else:
            mismatches.append((s, a_opt, a_trn))

    accuracy = correct / n_states
    per_action_acc = {}
    for a, cnt in per_action.items():
        per_action_acc[a] = per_action_correct[a] / cnt if cnt > 0 else np.nan

    return {
        "n_states": n_states,
        "correct": correct,
        "accuracy": accuracy,
        "per_action_counts": dict(per_action),
        "per_action_accuracy": per_action_acc,
        "mismatches": mismatches,  # list (state, optimal_action, agent_action)
    }






### von chatgpt


# 1) Optimalen Q-Vektor je Zustand (für Advantage/Confusion)
def optimal_qs(env_unwrapped, V, gamma=0.99):
    nS, nA = env_unwrapped.observation_space.n, env_unwrapped.action_space.n
    Q = np.zeros((nS, nA), dtype=np.float64)
    for s in range(nS):
        for a in range(nA):
            q = 0.0
            for p, s2, r, _ in env_unwrapped.P[s][a]:
                q += p * (r + gamma * V[s2])
            Q[s, a] = q
    return Q

# 2) Menge optimaler Aktionen pro Zustand (Ties zulassen)
def optimal_action_sets_from_Q(Q, tol=1e-9):
    best = Q.max(axis=1, keepdims=True)
    return [set(np.where(np.abs(Q[s] - best[s, 0]) <= tol)[0]) for s in range(Q.shape[0])]


@torch.no_grad()
def agent_action_onehot(agent, s, n_states=16):
    x = np.eye(n_states, dtype=np.float32)[s]  # (16,)
    a, _ = agent.predict(x, deterministic=True)
    return int(a)

# 3) Agent-Aktion je Zustand (deterministisch, ohne Epsilon)
@torch.no_grad()
def agent_action(agent, s: int):
    obs = np.array(s, dtype=np.int64)
    a, _ = agent.predict(obs, deterministic=True)
    return int(a)

# 4) Bias-/Konfusions-Analyse
def policy_bias_report(env_unwrapped, V_opt, agent, gamma=0.99, tol=1e-9):
    nS = env_unwrapped.observation_space.n
    nA = env_unwrapped.action_space.n

    Qstar = optimal_qs(env_unwrapped, V_opt, gamma=gamma)
    A_sets = optimal_action_sets_from_Q(Qstar, tol=tol)

    # Häufigkeiten
    opt_counts = np.zeros(nA, dtype=int)
    agent_counts = np.zeros(nA, dtype=int)

    # Confusion: Zeile = optimal (bei Ties: verteilt), Spalte = agent
    confusion = np.zeros((nA, nA), dtype=int)

    # Advantage-Gap bei Fehlern
    gaps = []

    correct = 0
    for s in range(nS):
        # optimale Aktionen (ggf. mehrere)
        Aopt = A_sets[s]
        # distribute one count gleichmäßig auf alle optimalen Aktionen
        for a in Aopt:
            opt_counts[a] += 1

        a_hat = agent_action(agent, s)
        agent_counts[a_hat] += 1

        if a_hat in Aopt:
            correct += 1
            # Confusion: verteilt auf alle optimalen Aktionen
            for a in Aopt:
                confusion[a, a_hat] += 1
        else:
            # Fehler: wähle einen „repräsentativen“ optimalen a* (z. B. beliebigen)
            a_star = next(iter(Aopt))
            confusion[a_star, a_hat] += 1
            # Advantage-Gap
            gap = Qstar[s, a_star] - Qstar[s, a_hat]
            gaps.append(float(gap))

    car = correct / nS
    # Normierte Verteilungen
    opt_dist = opt_counts / opt_counts.sum() if opt_counts.sum() > 0 else np.ones(nA)/nA
    agent_dist = agent_counts / agent_counts.sum() if agent_counts.sum() > 0 else np.ones(nA)/nA

    # KL(opt || agent): wie sehr der Agent von der optimalen Aktionsmarginalen abweicht
    eps = 1e-12
    kl = float(np.sum(opt_dist * (np.log(opt_dist + eps) - np.log(agent_dist + eps))))

    # Over-/Underuse pro Aktion (Agent vs. Optimum)
    overuse = agent_dist - opt_dist  # >0 = Agent nutzt Aktion häufiger als Optimum

    report = {
        "CAR_ties_allowed": car,
        "optimal_action_marginal": opt_dist.tolist(),
        "agent_action_marginal": agent_dist.tolist(),
        "overuse_per_action": overuse.tolist(),
        "KL_opt_vs_agent": kl,
        "confusion_matrix": confusion.tolist(),  # Zeile optimal, Spalte agent
        "avg_advantage_gap_on_errors": (float(np.mean(gaps)) if len(gaps) else 0.0),
        "n_states": nS,
    }
    return report

In [3]:
if __name__ == "__main__":
    # ==== Feste Parameter ====
    ENV_NAME = "FrozenLake-v1"   
    IS_SLIPPERY = False           
    CHECKPOINT_PATH = r"/Users/abdullah/Downloads/results/FrozenLake-v1/1.0M/Custom_DQN_FrozenLake_05bp_ebql_3qh/models/seed_0"  
    DEVICE = "cpu"
    SEED = 0
    GAMMA = 0.99
    THETA = 1e-8
    PRINT_MISMATCHES = True


    dp_env = gym.make(ENV_NAME, is_slippery=IS_SLIPPERY) if ENV_NAME == "FrozenLake-v1" else gym.make(ENV_NAME)
    env_unwrapped = dp_env.unwrapped 
    policy_opt, V_opt = value_iteration(env_unwrapped, gamma=GAMMA, theta=THETA)
    dp_env.close()

    
    # 2) Agent bauen & laden
    agent = make_agent(device=DEVICE, seed=SEED)
    agent.load(CHECKPOINT_PATH)

    # 3) Evaluieren
    metrics = evaluate_correct_action_rate(policy_opt, agent)

    # 4) Report
    print(f"=== Correct Action Rate vs Optimal Policy ({ENV_NAME}) ===")
    print(f"States:              {metrics['n_states']}")
    print(f"Correct matches:     {metrics['correct']}")
    print(f"Correct Action Rate: {metrics['accuracy']:.6f}")

    # print("\nPer-action counts (optimal policy frequency):")
    # for a, cnt in sorted(metrics["per_action_counts"].items()):
    #     print(f"  action {a}: {cnt}")

    # print("\nPer-action accuracy (agent matches optimal when optimal==a):")
    # for a, acc in sorted(metrics["per_action_accuracy"].items()):
    #     print(f"  action {a}: {acc:.6f}")

    # if PRINT_MISMATCHES:
    #     print("\nMismatches (state, optimal_action, agent_action):")
    #     for s, a_opt, a_trn in metrics["mismatches"]:
    #         print(s, a_opt, a_trn)


    print("\n=== Detailed Bias/Confusion Report ===")
    bias = policy_bias_report(dp_env.unwrapped, V_opt, agent, gamma=GAMMA, tol=1e-9)

    print("\n=== Policy Bias Report ===")
    print(f"CAR (ties allowed):       {bias['CAR_ties_allowed']:.6f}")
    print(f"KL(opt || agent):         {bias['KL_opt_vs_agent']:.6f}")
    print(f"Ø Advantage-Gap (Fehler): {bias['avg_advantage_gap_on_errors']:.6f}")
    print("Optimal marginal:", bias["optimal_action_marginal"])
    print("Agent   marginal:", bias["agent_action_marginal"])
    print("Overuse per action (agent - optimal):", bias["overuse_per_action"])
    print("Confusion [rows=optimal, cols=agent]:")
    for row in bias["confusion_matrix"]:
        print(row)


Value Iteration on FrozenLake-v1, states: 16, actions: 4
QNetwork(
  (encoder): MLP(
    (net): Sequential(
      (0): Linear(in_features=1, out_features=64, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU(inplace=True)
    )
  )
  (q_heads): ModuleList(
    (0-2): 3 x Linear(in_features=64, out_features=4, bias=True)
  )
)
QNetwork(
  (encoder): MLP(
    (net): Sequential(
      (0): Linear(in_features=1, out_features=64, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU(inplace=True)
    )
  )
  (q_heads): ModuleList(
    (0-2): 3 x Linear(in_features=64, out_features=4, bias=True)
  )
)
-----
=== Correct Action Rate vs Optimal Policy (FrozenLake-v1) ===
States:              16
Correct matches:     4
Correct Action Rate: 0.250000

=== Detailed Bias/Confusion Report ===

=== Policy Bias Report ===
CAR (ties allowed):       0.437500
KL(opt || agent

•	CAR (ties allowed): Anteil der Zustände, in denen deine Policy eine der optimalen Aktionen wählt, wenn mehrere gleich gut sind.

•	KL(opt || agent): Wie stark die Aktionsverteilung deiner Policy von der optimalen abweicht; höher heißt schiefer.
•	Ø Advantage-Gap (Fehler): Mittlerer Wertverlust deiner gewählten falschen Aktion gegenüber einer optimalen in demselben Zustand.
•	Optimal marginal: Häufigkeit jeder Aktion unter der optimalen Policy, über alle Zustände gemittelt.
•	Agent marginal: Häufigkeit jeder Aktion unter deiner Policy, über alle Zustände gemittelt.
•	Overuse per action (agent − optimal): Positive Werte zeigen, welche Aktionen dein Agent zu oft nutzt und negative, welche er zu selten nutzt.
•	Confusion-Matrix (rows=optimal, cols=agent): Für jede optimale Aktion zeigt die Zeile, wie oft dein Agent stattdessen jede mögliche Aktion gewählt hat.